<a href="https://colab.research.google.com/github/Yousef-Tarek511/calories-burn-ml-predictor/blob/main/Calories_Burnt_linear_regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Multiple Linear Regression

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd


## Importing the dataset

In [ ]:
hourly_calories = pd.read_csv('hourlyCalories_merged.csv')
hourly_intensities = pd.read_csv('hourlyIntensities_merged.csv')
hourly_steps = pd.read_csv('hourlySteps_merged.csv')
heartrate = pd.read_csv('heartrate_seconds_merged.csv')
weight = pd.read_csv('weightLogInfo_merged.csv')
sleep = pd.read_csv('minuteSleep_merged.csv')

df_hourly = hourly_calories.merge(hourly_intensities, on=['Id', 'ActivityHour']).merge(hourly_steps, on=['Id', 'ActivityHour'])
df_hourly['ActivityHour'] = pd.to_datetime(df_hourly['ActivityHour'], format='%m/%d/%Y %I:%M:%S %p')
df_hourly.drop(columns=['AverageIntensity'], inplace=True)

heartrate['Time'] = pd.to_datetime(heartrate['Time'], format='%m/%d/%Y %I:%M:%S %p')
heartrate['ActivityHour'] = heartrate['Time'].dt.floor('h')
hr_hourly = heartrate.groupby(['Id', 'ActivityHour'])['Value'].agg(
    AvgHeartRate='mean',
    MaxHeartRate='max',
    MinHeartRate='min'
).reset_index()

print("Duplicates in minuteSleep:", sleep.duplicated().sum())
sleep.drop_duplicates(inplace=True)
sleep['date'] = pd.to_datetime(sleep['date'], format='%m/%d/%Y %I:%M:%S %p')
sleep['ActivityHour'] = sleep['date'].dt.floor('h')
sleep_hourly = sleep.groupby(['Id', 'ActivityHour']).size().reset_index(name='SleepMinutes')

weight_user = weight.drop(columns=['Fat']).groupby('Id')[['WeightKg', 'BMI']].mean().reset_index()

dataset = df_hourly.merge(hr_hourly, on=['Id', 'ActivityHour'], how='inner')
dataset = dataset.merge(sleep_hourly, on=['Id', 'ActivityHour'], how='left')
dataset = dataset.merge(weight_user, on='Id', how='left')

dataset['SleepMinutes'] = dataset['SleepMinutes'].fillna(0)
dataset['Hour'] = dataset['ActivityHour'].dt.hour
dataset['DayOfWeek'] = dataset['ActivityHour'].dt.day_name()

dataset = dataset[
    ['StepTotal', 'TotalIntensity', 'AvgHeartRate', 'MaxHeartRate', 'MinHeartRate',
     'SleepMinutes', 'WeightKg', 'BMI', 'Hour', 'DayOfWeek', 'Calories']
]

dataset.head()

Duplicates in minuteSleep: 525


,StepTotal,TotalIntensity,AvgHeartRate,MaxHeartRate,MinHeartRate,SleepMinutes,WeightKg,BMI,Hour,DayOfWeek,Calories
0,123,5,100.326531,112,87,0.0,NaN,NaN,7,Friday,78
1,440,19,71.396419,111,53,0.0,NaN,NaN,8,Friday,115
2,4994,146,121.205323,158,68,0.0,NaN,NaN,9,Friday,427
3,2926,90,97.129771,119,68,0.0,NaN,NaN,10,Friday,248
4,0,0,90.000000,90,90,0.0,NaN,NaN,11,Friday,62


### Information about dataset

In [ ]:
dataset.shape

(2532, 11)

In [ ]:
dataset.info()

<class 'pandas.DataFrame'>
RangeIndex: 2532 entries, 0 to 2531
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   StepTotal       2532 non-null   int64  
 1   TotalIntensity  2532 non-null   int64  
 2   AvgHeartRate    2532 non-null   float64
 3   MaxHeartRate    2532 non-null   int64  
 4   MinHeartRate    2532 non-null   int64  
 5   SleepMinutes    2532 non-null   float64
 6   WeightKg        1012 non-null   float64
 7   BMI             1012 non-null   float64
 8   Hour            2532 non-null   int32  
 9   DayOfWeek       2532 non-null   str    
 10  Calories        2532 non-null   int64  
dtypes: float64(4), int32(1), int64(5), str(1)
memory usage: 207.8 KB


In [ ]:
dataset.describe().T

,count,mean,std,min,25%,50%,75%,max
StepTotal,2532.0,517.361374,868.834649,0.000000,14.000000,219.500000,660.250000,10565.000000
TotalIntensity,2532.0,19.053318,25.615503,0.000000,1.000000,11.000000,26.000000,180.000000
AvgHeartRate,2532.0,76.930542,14.464102,43.750799,66.930960,75.037117,85.587581,161.930168
MaxHeartRate,2532.0,100.024487,21.265220,52.000000,84.000000,99.000000,113.000000,185.000000
MinHeartRate,2532.0,62.383886,9.672162,36.000000,56.000000,61.000000,68.000000,145.000000
SleepMinutes,2532.0,13.593997,24.317362,0.000000,0.000000,0.000000,9.000000,60.000000
WeightKg,1012.0,68.073205,8.700085,61.864286,61.864286,63.400002,69.400002,85.755556
BMI,1012.0,25.188435,1.076806,24.155000,24.155000,24.770000,25.677778,27.139999
Hour,2532.0,12.100711,6.456226,0.000000,7.000000,12.000000,17.250000,23.000000
Calories,2532.0,117.505529,75.949439,47.000000,71.000000,94.000000,139.250000,933.000000


In [ ]:
dataset.describe(include=['object', 'string']).T

,count,unique,top,freq
DayOfWeek,2532,7,Friday,455


### Check Missing Value

In [ ]:
dataset.isnull().sum()

,0
StepTotal,0
TotalIntensity,0
AvgHeartRate,0
MaxHeartRate,0
MinHeartRate,0
SleepMinutes,0
WeightKg,1520
BMI,1520
Hour,0
DayOfWeek,0


### Check Duplicates

In [ ]:
dataset.duplicated().sum()


np.int64(0)

In [ ]:
dataset.drop_duplicates(inplace=True)

In [ ]:
dataset.to_csv('cleaned_dataset.csv',index=False)